<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Score Langfuse Traces with Hermes Rubric" sidebarTitle: "Hermes Rubric" logo: "/images/integrations/hermes_rubric_icon.svg" description: "Learn how to use Hermes Rubric's evidence-first LLM-as-judge scoring to grade Langfuse traces and push per-dimension scores back onto them." category: "Integrations" -->

# Score Langfuse Traces with Hermes Rubric

[Hermes Rubric](https://github.com/hermes-labs-ai/hermes-rubric) is an evidence-first LLM-as-judge scoring library. Given a target artifact (an agent output, a PR diff, a cold email, ...), an intent, and optional context, it synthesizes or accepts a rubric, collects cited evidence for each dimension, and scores only against that accepted evidence. Score responses that fail validation raise instead of silently becoming a fallback score.

This notebook shows how to grade a Langfuse trace's output with `hermes_rubric.assess()` and attach the resulting per-dimension scores back onto that trace using the Langfuse Python SDK's `score()` call, so the scores show up next to the trace in the Langfuse UI.

## Step 1: Install dependencies

In [ ]:
%pip install "hermes-rubric[openai]" langfuse openai -q

## Step 2: Configure the Langfuse SDK

Set your Langfuse API keys. Get these by signing up for [Langfuse Cloud](https://cloud.langfuse.com/) or by [self-hosting Langfuse](https://langfuse.com/self-hosting).

In [ ]:
import os

os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"  # or your self-hosted URL
os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 3: Grade a traced generation and score the trace

In [ ]:
from langfuse import get_client
from hermes_rubric import FeedbackPolicy, assess

langfuse = get_client()

with langfuse.start_as_current_span(name="support-reply") as span:
    task_context = "Customer asked whether the Pro plan includes SSO."
    agent_output = (
        "Yes, SSO (SAML and OIDC) is included on the Pro plan at no extra cost; "
        "see the Security settings page to configure it."
    )
    span.update(input=task_context, output=agent_output)

    result = assess(
        target=agent_output,
        intent="Answer accurately and support material claims with checkable evidence.",
        context=task_context,
        target_type="agent-output",
        backend="openai-sdk",
    )

    # Push the aggregate score onto the trace
    langfuse.score_current_trace(
        name="hermes-rubric-aggregate",
        value=result.aggregate,
        comment=result.coverage.status,
    )

    # Push each rubric dimension as its own score, citing the coverage source
    for dim in result.per_dim_scores:
        langfuse.score_current_trace(
            name=f"hermes-rubric-{dim.name}",
            value=dim.score,
            comment=dim.rationale if hasattr(dim, "rationale") else None,
        )

langfuse.flush()

The aggregate score and per-dimension scores now appear on the trace in the Langfuse UI, alongside any other scores (human annotation, other evaluators) attached to the same trace. Because `assess()` raises `AssessmentError` on a malformed or unresolvable score rather than returning a silent fallback value, a failed grading pass will surface as an exception here instead of writing a misleading score onto the trace.

See the [Hermes Rubric README](https://github.com/hermes-labs-ai/hermes-rubric#readme) for backend options (OpenAI, local Ollama, Claude Code, and other backends), the coverage/hedging model, and reusable frozen rubrics for grading many traces against the same dimensions.